# Used Vehicles Exploratory Data Analysis

This notebook analyzes the external `vehicles.csv` dataset. Provide the CSV using `USED_CARS_CSV`, `data/vehicles.csv`, `vehicles.csv`, or the Kaggle input path documented in the README.

In [ ]:
from pathlib import Path
import os

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
candidate_paths = []
if os.environ.get("USED_CARS_CSV"):
    candidate_paths.append(Path(os.environ["USED_CARS_CSV"]).expanduser())
candidate_paths.extend([
    Path("data/vehicles.csv"),
    Path("vehicles.csv"),
    Path("/kaggle/input/craigslist-carstrucks-data/vehicles.csv"),
])
csv_path = next((path for path in candidate_paths if path.is_file()), None)
if csv_path is None:
    checked_paths = ", ".join(str(path) for path in candidate_paths)
    raise FileNotFoundError(
        "vehicles.csv was not found. Set USED_CARS_CSV or place the file at "
        f"one of these locations: {checked_paths}"
    )
print(f"Using dataset: {csv_path}")

In [ ]:
df = pd.read_csv(csv_path)
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

### Missing-value handling

Missing values are left as missing rather than replaced globally with a mean or the most frequent category. This avoids creating synthetic listings or distorting category counts. Check missingness before interpreting each analysis; pandas summaries and groupings omit missing values by default.

In [ ]:
df.nunique(dropna=False).sort_values(ascending=False)

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
df.duplicated().sum()

In [ ]:
df.dtypes

In [ ]:
df['posting_date'] = pd.to_datetime(df['posting_date'], errors='coerce')

In [ ]:
df.head(3)

# EDA

# What are the most popular type?

In [ ]:
print(df.columns)

In [ ]:
df['type'].value_counts()

In [ ]:
df['type'].value_counts().plot(kind='pie',autopct='%1.1f%%');

In [ ]:
sns.countplot(x='type', data=df, order=df['type'].value_counts().index)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# What is the distribution of the number of cars according to title status?

In [ ]:
plt.figure(figsize=(10, 6))
df['title_status'].value_counts().plot(kind='bar', color='orange')
plt.title('Distribution of Cars by Title Status')
plt.xlabel('Title Status')
plt.ylabel('Count')
plt.show()


# What is the most common condition for cars?

In [ ]:
most_common = df['condition'].mode()[0]
print(f"The most common case: {most_common}")

In [ ]:
plt.figure(figsize=(8, 5))
df['condition'].value_counts().plot(kind='bar', color='red')
plt.title('Most Common Car Conditions')
plt.xlabel('Condition')
plt.ylabel('Count')
plt.show()

# What is the distribution of data over time?

In [ ]:
posting_year = df['posting_date'].dt.year.dropna().astype(int)
yearly_distribution = posting_year.value_counts().sort_index()
plt.figure(figsize=(14, 8))
yearly_distribution.plot(kind='area', color='green', alpha=0.4)
plt.title('Car Listings by Posting Year')
plt.xlabel('Posting Year')
plt.ylabel('Number of Listings')
plt.show()

# What is the relationship between distance traveled and price?

In [ ]:
plt.figure(figsize=(12, 8))
df.groupby('odometer')['price'].mean().plot(kind='line', color='blue')
plt.title('Average Price vs Odometer')
plt.xlabel('Odometer')
plt.ylabel('Average Price')
plt.grid(True)
plt.show()


# How do prices compare between fuel types?

In [ ]:
fuel_price_comparison = df.groupby('fuel')['price'].mean().sort_values()
plt.figure(figsize=(12, 8))
fuel_price_comparison.plot(kind='bar', color='yellow')
plt.title('Average Price by Fuel Type')
plt.xlabel('Fuel Type')
plt.ylabel('Average Price')
plt.show()


# Thanks